# 한국마사회 구매성향분석 (2022-01 ~ 2024-04)

한국마사회 공공데이터포털 OpenAPI(`pchTndcyAnal`, https://www.data.go.kr/data/15154736/openapi.do )에서 월별·시행경마장·판매경마장 조합으로 수집한 마권 구매성향 데이터를 분석합니다.

- 데이터: `raw/한국마사회_구매성향분석_202201-202404.csv` (28개월 × 3개 시행경마장 × 3개 판매경마장 = 252행)
- 컬럼: `performRacecourse`(시행경마장, 경주가 열리는 곳) · `saleRacecourse`(판매경마장, 마권이 팔린 채널) · `raceMonth` · `saleCount`(발매건수) · `saleValue`(발매금액, 원)

> 이 API는 (경주년월 × 시행경마장 × 판매경마장) 조합 하나당 결과 1건만 반환하는 구조라, 여러 조합을 반복 호출해 CSV로 모았습니다. `04:양천`은 실제 경주가 열리지 않는 장외발매소라 조합에서 제외했습니다(호출 시 0건).

**요약 결론**: 경주 개최 비중은 서울 50.6%·부산경남 28.9%·제주 20.5%로 비교적 고르게 분산되어 있지만, 실제 마권 판매는 **판매경마장(채널) 기준 서울이 전체의 90.8%를 독점**합니다. 특히 부산경남 경주는 95%가 '서울' 판매 채널을 통해 팔리는 등, 경주 개최지와 판매 채널이 완전히 분리되어 있습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['axes.unicode_minus'] = False
# plt.rc('font', family='NanumGothic')  # 한글 폰트가 있다면 주석 해제 (Linux/Windows)
# plt.rc('font', family='AppleGothic')  # macOS

df = pd.read_csv('raw/한국마사회_구매성향분석_202201-202404.csv')
df['year'] = df['raceMonth'] // 100
df['month'] = df['raceMonth'] % 100
df['is_self'] = df['performRacecourse'] == df['saleRacecourse']

print(df.shape)
df.head()

## 1. 데이터 확인

결측치, 기간, 조합 개수를 확인합니다.

In [ ]:
print("결측치:")
print(df.isnull().sum())
print()
print(f"기간: {df['raceMonth'].min()} ~ {df['raceMonth'].max()} ({df['raceMonth'].nunique()}개월)")
print(f"시행경마장: {sorted(df['performRacecourse'].unique())}")
print(f"판매경마장: {sorted(df['saleRacecourse'].unique())}")
print(f"전체 발매금액: {df['saleValue'].sum():,}원 ({df['saleValue'].sum()/1e12:.2f}조원)")
print(f"전체 발매건수: {df['saleCount'].sum():,}건")

## 2. 월별 추이

2022~2023년은 온전한 연도, 2024년은 1~4월만 존재합니다. 연도 비교는 월평균 기준으로 봅니다.

In [ ]:
monthly = df.groupby('raceMonth')['saleValue'].sum()

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(ax=ax, color='#12897B', marker='o', markersize=3)
ax.set_title('월별 총 발매금액 추이')
ax.set_ylabel('발매금액(원)')
plt.tight_layout()
plt.show()

year_avg = df.groupby('year').apply(lambda g: g.groupby('raceMonth')['saleValue'].sum().mean())
print("연도별 월평균 발매금액(억원):")
print((year_avg / 1e8).round(1))
print(f"\n2023 vs 2022 YoY(월평균): {(year_avg[2023]/year_avg[2022]-1)*100:+.1f}%")

same_months = df[df['month'].isin([1, 2, 3, 4])].groupby('year')['saleValue'].sum()
print(f"2024 vs 2023 YoY(1~4월 동기간): {(same_months[2024]/same_months[2023]-1)*100:+.1f}%")

**인사이트**: 2022→2023년은 월평균 +1.6%로 거의 보합, 2024년 1~4월도 전년 동기 대비 -1.8%로 큰 변화가 없습니다. 이 기간 동안 마권 판매 규모는 전반적으로 안정적입니다 — 아래에서 보듯 진짜 특징은 시간 추이가 아니라 **판매 채널의 쏠림**입니다.

## 3. 계절성

In [ ]:
season = df.groupby('month')['saleValue'].sum()

fig, ax = plt.subplots(figsize=(8, 4))
top3 = set(season.sort_values(ascending=False).index[:3])
colors = ['#0B7A69' if m in top3 else '#AEE0D3' for m in season.index]
season.plot.bar(ax=ax, color=colors)
ax.set_title('월별(계절) 발매금액 합산 (2022~2024 누적)')
ax.set_xlabel('월')
plt.tight_layout()
plt.show()

**인사이트**: 3~4월(봄 시즌 개막)과 7월에 발매금액이 높고, 9·11월과 2월(설 연휴 영향)이 낮습니다 — 경마 시즌 개막·황금연휴와 맞물린 계절성으로 보입니다.

## 4. 시행경마장 vs 판매경마장 — 채널 쏠림

⚠️ 이 데이터셋의 핵심 발견입니다. 경주가 실제로 열리는 곳(시행경마장)과 마권이 팔리는 채널(판매경마장)을 분리해서 봅니다.

In [ ]:
perf_share = df.groupby('performRacecourse')['saleValue'].sum().sort_values(ascending=False)
sale_share = df.groupby('saleRacecourse')['saleValue'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
(perf_share / perf_share.sum() * 100).plot.bar(ax=axes[0], color='#12897B')
axes[0].set_title('시행경마장별 경주 비중 (%)')
axes[0].set_ylabel('%')

(sale_share / sale_share.sum() * 100).plot.bar(ax=axes[1], color='#C2501F')
axes[1].set_title('판매경마장(채널)별 발매 비중 (%)')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

print("시행경마장별 비중(%):")
print((perf_share / perf_share.sum() * 100).round(1))
print("\n판매경마장(채널)별 비중(%):")
print((sale_share / sale_share.sum() * 100).round(1))

**인사이트**: 경주 개최 비중은 서울 50.6%·부산경남 28.9%·제주 20.5%로 비교적 고르지만, 판매 채널은 **'서울' 채널이 전체 발매금액의 90.8%를 독점**합니다. 부산경남·제주 채널은 각각 4.4%·4.8%에 불과합니다.

### 4.1 경마장별 자체판매 비중

각 경마장에서 열리는 경주가, 같은 경마장의 판매 채널을 통해 팔리는 비중을 봅니다.

In [ ]:
self_cross = df.groupby(['performRacecourse', 'is_self'])['saleValue'].sum().unstack()
self_cross.columns = ['타채널 판매', '자체채널 판매']
self_cross['자체채널 비중(%)'] = (self_cross['자체채널 판매'] / (self_cross['자체채널 판매'] + self_cross['타채널 판매']) * 100).round(1)
self_cross

**인사이트**: 서울 경주는 91.9%가 서울 채널에서 자체 판매되는 반면, **부산경남 경주는 95.0%, 제주 경주는 90.9%가 '서울' 채널을 통해 팔립니다**. 즉 지역 경마장이 스스로 개최하는 경주조차 자기 채널보다 서울 채널 의존도가 훨씬 높습니다 — 경주 개최지와 마권 판매 채널이 사실상 분리되어 있습니다.

### 4.2 판매경마장별 평균 구매단가

In [ ]:
avg_bet = df.groupby('saleRacecourse').apply(lambda g: g['saleValue'].sum() / g['saleCount'].sum())
avg_bet = avg_bet.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
avg_bet.plot.bar(ax=ax, color='#12897B')
ax.set_title('판매경마장(채널)별 평균 구매단가')
ax.set_ylabel('원/건')
plt.tight_layout()
plt.show()

avg_bet.round(0)

**인사이트**: 제주 채널 이용자의 평균 구매단가(16,469원/건)가 서울(11,626원)·부산경남(8,458원)보다 뚜렷하게 높습니다. 판매 건수는 적지만 건당 구매 규모가 큰 채널입니다.

## 5. 개선 액션 플랜

### 즉시 실행 (0-3개월)
1. **지역 판매 채널 활성화 진단** — 부산경남·제주 경주의 95%·91%가 서울 채널로 판매되는 원인(오프라인 발매소 접근성, 온라인 채널 지역 배정 방식 등)을 진단합니다. *(근거: 부산경남 자체채널 비중 5.0%, 제주 9.1%)*
2. **서울 채널 부하 모니터링** — 전체 발매금액의 90.8%가 몰리는 서울 채널의 시스템·발매소 처리 용량을 점검합니다.

### 중기 과제 (3-12개월)
1. **채널별 맞춤 프로모션** — 제주 채널처럼 건당 구매단가가 높은 채널은 고액 구매자 대상 서비스를, 부산경남처럼 자체 채널 비중이 낮은 지역은 접근성 개선을 검토합니다.
2. **계절 성수기 대비 인력·시스템 배치** — 3~4월, 7월 성수기 직전에 발매 시스템·인력을 선제적으로 확충합니다.

### 장기 과제 (1년 이상)
1. **경주 개최지-판매 채널 연계 전략 재검토** — 지역 경마장이 스스로 개최하는 경주의 판매 수익 대부분이 서울 채널로 귀속되는 구조가 지역 경마 산업의 지속가능성에 미치는 영향을 검토합니다.

## 6. 데이터 한계 및 방법론 노트

- 이 데이터는 시행경마장 3곳(서울·제주·부산경남) × 판매경마장 3곳의 조합만 존재합니다(04:양천은 장외발매소라 경주가 열리지 않아 조합에서 제외). 판매 채널이 3개로 광역 집계되어 있어, 실제 개별 발매소 단위의 지역 분포는 알 수 없습니다.
- 2024년은 1~4월 데이터만 수집했습니다(API 호출 시점 기준). 연도 비교는 총합이 아닌 동월 비교·월평균 기준으로 계산했습니다.
- '취소건수', '환불금액' 등 공공데이터포털 설명에 언급된 필드는 이 API 응답에는 포함되어 있지 않았습니다(발매건수·발매금액만 제공).
- 본 분석은 서술 통계 기반이며, 제시된 관계는 상관관계로 인과관계를 증명하지 않습니다.